# CNN malware appliqué à nos modèles Python compilés

Le CNN du TP de cybersécurité classe des binaires Windows en **9 familles de malware**
(jeu Microsoft BIG 2015), à partir d'une image construite avec les octets du programme.
Ce notebook le réentraîne, puis lui soumet **les classifieurs de nos quatre TP de
caractérisation d'images, compilés en `.exe` avec PyInstaller**.

| Fichier soumis | Contenu |
|---|---|
| `modele01_lbp.exe` … `modele04_multiechelle.exe` | nos classifieurs LBP, compilés |
| `temoin_pyinstaller.exe` | un programme vide (`print`), compilé de la même façon |
| `temoin_notepad.exe`, `temoin_cmd.exe` | logiciels Windows sains, signés Microsoft |

Le témoin PyInstaller sert à séparer deux effets : s'il reçoit la même réponse que nos
modèles, le CNN réagit à l'outil de compilation et non à notre code.

**Limite à garder en tête.** Le CNN n'a pas de classe « logiciel sain » : il répond
toujours par l'une des 9 familles. On ne lit donc pas sa réponse comme « virus / pas
virus », mais on compare sa **confiance** sur nos fichiers à sa confiance sur de vrais
malwares.

**Avant de lancer :** `Donnees1.zip` et `executables.zip` dans ton Google Drive, et le GPU
activé (*Exécution > Modifier le type d'exécution > T4 GPU*). Ensuite *Exécution > Tout
exécuter*.

In [ ]:
import glob, hashlib, io, os, struct, time, zipfile
import numpy as np
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPU :", gpus or "AUCUN")
if not gpus:
    print("*** Active le GPU : Execution > Modifier le type d'execution > T4 GPU ***")

from google.colab import drive
drive.mount("/content/drive")

## 1. Données et exécutables

In [ ]:
def trouver(nom):
    """Cherche d'abord dans MonDrive/cyber/, puis dans tout le Drive."""
    defaut = "/content/drive/MyDrive/cyber/" + nom
    if os.path.exists(defaut):
        return defaut
    candidats = glob.glob("/content/drive/MyDrive/**/" + nom, recursive=True)
    assert candidats, nom + " introuvable dans ton Google Drive"
    return candidats[0]

DATA_DIR, EXE_DIR = "/content/data", "/content/executables"
for nom, cible in (("Donnees1.zip", DATA_DIR), ("executables.zip", EXE_DIR)):
    if not os.path.isdir(cible):
        chemin = trouver(nom)
        print("Decompression de", chemin)
        zipfile.ZipFile(chemin).extractall(cible)

EXECUTABLES = sorted(glob.glob(EXE_DIR + "/**/*.exe", recursive=True))
print(len(EXECUTABLES), "executables :", [os.path.basename(e) for e in EXECUTABLES])

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

IMG, N_CLASSES = 128, 9
NOMS = ["Ramnit", "Lollipop", "Kelihos_ver3", "Vundo", "Simda",
        "Tracur", "Kelihos_ver1", "Obfuscator.ACY", "Gatak"]

def charger_split(split):
    fichiers, labels = [], []
    for c in range(1, N_CLASSES + 1):
        fs = sorted(glob.glob(os.path.join(DATA_DIR, split, str(c), "*.jpg")))
        fichiers += fs
        labels += [c - 1] * len(fs)
    X = np.zeros((len(fichiers), IMG, IMG, 1), dtype=np.uint8)
    def lire(i):
        im = Image.open(fichiers[i]).convert("L").resize((IMG, IMG), Image.BILINEAR)
        X[i, :, :, 0] = np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(lire, range(len(fichiers))))
    return X, np.array(labels, dtype=np.int32)

Xtr, ytr = charger_split("train")
Xva, yva = charger_split("validation")
Xte, yte = charger_split("test")
print("train", Xtr.shape, "| validation", Xva.shape, "| test", Xte.shape)

## 2. Entraînement du CNN

Configuration retenue : la meilleure du TP sur le jeu A, soit l'architecture du cours
corrigée (normalisation des entrées, `class_weight`, arrêt anticipé) **sans dropout**.

In [ ]:
from tensorflow.keras import Input, Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(42)
modele = Sequential([
    Input((IMG, IMG, 1)), layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(50, activation="relu"),
    layers.Dense(N_CLASSES, activation="softmax"),
])
modele.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])

poids = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=ytr)
t0 = time.time()
modele.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=12, batch_size=32,
           class_weight=dict(enumerate(poids)), verbose=2,
           callbacks=[EarlyStopping(patience=3, restore_best_weights=True),
                      ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5)])

proba_test = modele.predict(Xte, batch_size=256, verbose=0)
pred_test = proba_test.argmax(1)
print("\nEntrainement : %.0f s" % (time.time() - t0))
print("TEST  accuracy = %.4f | F1 macro = %.4f"
      % (accuracy_score(yte, pred_test), f1_score(yte, pred_test, average="macro")))

## 3. Confiance de référence sur de vrais malwares

Pour chaque image, la **confiance** est la probabilité de la famille choisie. On la mesure
sur les malwares du jeu de test : c'est l'étalon auquel comparer nos fichiers. Le seuil
retenu est le 5e centile : 95 % des vrais malwares sont classés avec une confiance au moins
égale.

In [ ]:
conf_test = proba_test.max(1)
SEUIL = float(np.percentile(conf_test, 5))
print("Confiance sur les %d malwares de test :" % len(conf_test))
print("  mediane         %.3f" % np.median(conf_test))
print("  5e centile      %.3f   <- seuil" % SEUIL)
print("  minimum         %.3f" % conf_test.min())

## 4. Conversion d'un `.exe` en image, identique au cours

La conversion reprend `BytesToImage.ipynb`, qui a produit les images du jeu. Deux détails
de ce pipeline décident de ce que le CNN voit d'un programme :

1. Les fichiers `.bytes` de BIG 2015 sont le vidage du programme **chargé en mémoire, à
   partir de sa première section** : l'en-tête PE est retiré, les octets indéterminés
   (`??`) valent 0.
2. `saveimg()` appelle `numpy.ndarray.resize((256, 256))`. Ce n'est **pas** un
   redimensionnement d'image : cette méthode **tronque** le tableau à ses 65 536 premiers
   éléments. L'image du cours est donc exactement **les 64 premiers Ko du programme**,
   par lignes de 256 octets, enregistrés en JPEG.

La suite applique le même chargement que pour le jeu (niveaux de gris, redimensionnement
bilinéaire en 128 x 128).

In [ ]:
FENETRE = 256 * 256

def sections_pe(donnees):
    assert donnees[:2] == b"MZ", "pas un executable Windows"
    pe = struct.unpack_from("<I", donnees, 0x3C)[0]
    assert donnees[pe:pe + 4] == b"PE\0\0", "en-tete PE introuvable"
    nb = struct.unpack_from("<H", donnees, pe + 6)[0]
    table = pe + 24 + struct.unpack_from("<H", donnees, pe + 20)[0]
    sections = []
    for i in range(nb):
        e = table + 40 * i
        nom = donnees[e:e + 8].rstrip(b"\0").decode("ascii", "replace")
        taille_virt, adresse, taille_brute, position = struct.unpack_from("<IIII", donnees, e + 8)
        sections.append((adresse, taille_virt, taille_brute, position, nom))
    return sorted(sections)

def fenetre_vue_par_le_cnn(donnees):
    sections = sections_pe(donnees)
    base, memoire = sections[0][0], bytearray(FENETRE)
    for adresse, taille_virt, taille_brute, position, _ in sections:
        debut = adresse - base
        if debut >= FENETRE:
            break
        n = min(taille_brute, taille_virt or taille_brute, FENETRE - debut)
        memoire[debut:debut + n] = donnees[position:position + n]
    return bytes(memoire)

def image_cnn(fenetre):
    tableau = np.frombuffer(fenetre, dtype=np.uint8).reshape(256, 256)
    tampon = io.BytesIO()
    Image.fromarray(tableau).save(tampon, "JPEG")          # comme saveimg()
    tampon.seek(0)
    im = Image.open(tampon).convert("L").resize((IMG, IMG), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)                  # comme charger_split()

def categorie(nom):
    if nom.startswith("modele"):
        return "notre modele"
    if nom.startswith("temoin_pyinstaller"):
        return "temoin PyInstaller"
    if nom.startswith("temoin"):
        return "temoin Windows sain"
    return "autre"

FICHIERS = []
for chemin in EXECUTABLES:
    donnees = open(chemin, "rb").read()
    fenetre = fenetre_vue_par_le_cnn(donnees)
    nom = os.path.basename(chemin)[:-4]
    FICHIERS.append(dict(nom=nom, categorie=categorie(nom),
                         taille_mo=len(donnees) / 2**20,
                         part_vue_pct=100 * FENETRE / len(donnees),
                         empreinte=hashlib.sha256(fenetre).hexdigest()[:12],
                         image=image_cnn(fenetre)))

print("%-24s %-20s %9s %10s  %s" % ("fichier", "categorie", "taille", "part vue", "empreinte des 64 Ko"))
for f in FICHIERS:
    print("%-24s %-20s %6.1f Mo %9.3f %%  %s"
          % (f["nom"], f["categorie"], f["taille_mo"], f["part_vue_pct"], f["empreinte"]))

groupes = {}
for f in FICHIERS:
    groupes.setdefault(f["empreinte"], []).append(f["nom"])
print()
for empreinte, noms in groupes.items():
    if len(noms) > 1:
        print("MEME IMAGE (octet pour octet) : " + ", ".join(noms))

## 5. Réponse du CNN sur chaque fichier

In [ ]:
import pandas as pd

X_exe = np.stack([f["image"] for f in FICHIERS])[..., None]
proba_exe = modele.predict(X_exe, verbose=0)

for f, p in zip(FICHIERS, proba_exe):
    ordre = p.argsort()[::-1]
    f["famille"] = NOMS[ordre[0]]
    f["confiance"] = float(p[ordre[0]])
    f["deuxieme"] = "%s (%.2f)" % (NOMS[ordre[1]], p[ordre[1]])
    f["rang_vs_malwares"] = float((conf_test < f["confiance"]).mean() * 100)
    f["verdict"] = ("ressemble a %s, aussi nettement qu'un vrai malware" % f["famille"]
                    if f["confiance"] >= SEUIL else "ne ressemble a aucune famille connue")

TABLE = pd.DataFrame(FICHIERS).drop(columns="image")
pd.set_option("display.width", 200)
print(TABLE[["nom", "categorie", "famille", "confiance", "deuxieme", "rang_vs_malwares",
             "verdict"]].to_string(index=False, float_format=lambda v: "%.3f" % v))
print("\nrang_vs_malwares : % des vrais malwares de test classes avec MOINS de confiance.")

In [ ]:
import matplotlib.pyplot as plt

COULEURS = {"notre modele": "#2a78d6", "temoin PyInstaller": "#eb6834",
            "temoin Windows sain": "#1baf7a", "autre": "#898781"}
os.makedirs("/content/sorties", exist_ok=True)

# Figure 1 : ce que voit le CNN, a cote d'un vrai malware de chaque famille
n = max(len(FICHIERS), N_CLASSES)
fig, axes = plt.subplots(2, n, figsize=(1.6 * n, 4.2))
for ax in axes.ravel():
    ax.axis("off")
for ax, f in zip(axes[0], FICHIERS):
    ax.imshow(f["image"], cmap="gray", vmin=0, vmax=255)
    ax.set_title("%s\n-> %s %.2f" % (f["nom"], f["famille"], f["confiance"]),
                 fontsize=7, color=COULEURS[f["categorie"]])
for c, ax in enumerate(axes[1][:N_CLASSES]):
    ax.imshow(Xte[np.where(yte == c)[0][0], :, :, 0], cmap="gray", vmin=0, vmax=255)
    ax.set_title("malware\n" + NOMS[c], fontsize=7)
fig.suptitle("En haut : nos fichiers, tels que le CNN les voit.  En bas : un vrai malware par famille.",
             fontsize=10, x=0.01, ha="left")
plt.tight_layout()
plt.savefig("/content/sorties/images_vues_par_le_cnn.png", dpi=130)
plt.show()

# Figure 2 : confiance de nos fichiers face a celle des vrais malwares
fig, ax = plt.subplots(figsize=(9, 0.45 * len(FICHIERS) + 1.8))
bas, haut = np.percentile(conf_test, [5, 95])
ax.axvspan(bas, haut, color="#e1e0d9", zorder=0)
ax.axvline(SEUIL, color="#52514e", linewidth=1, linestyle=(0, (4, 3)))
ax.annotate("90 % des vrais malwares", (bas + 0.01, len(FICHIERS) - 0.45),
            fontsize=8, color="#52514e")
for y, f in enumerate(FICHIERS):
    ax.scatter(f["confiance"], y, s=80, color=COULEURS[f["categorie"]], zorder=3,
               edgecolors="#fcfcfb", linewidths=1.5)
    ax.annotate(f["famille"], (f["confiance"], y), textcoords="offset points",
                xytext=(9, -3), fontsize=8, color="#52514e")
ax.set_yticks(range(len(FICHIERS)), [f["nom"] for f in FICHIERS])
ax.set_xlim(0, 1.12)
ax.set_ylim(-0.6, len(FICHIERS) - 0.1)
ax.set_xlabel("confiance du CNN (probabilite de la famille choisie)")
ax.set_title("Confiance du CNN : nos fichiers face aux vrais malwares (zone grise)",
             loc="left", fontsize=10, fontweight="bold")
for cote in ("top", "right"):
    ax.spines[cote].set_visible(False)
ax.grid(axis="x", color="#e1e0d9")
plt.tight_layout()
plt.savefig("/content/sorties/confiance.png", dpi=130)
plt.show()

## 6. Conclusion (générée à partir des résultats)

In [ ]:
modeles = [f for f in FICHIERS if f["categorie"] == "notre modele"]
temoin = next((f for f in FICHIERS if f["categorie"] == "temoin PyInstaller"), None)
sains = [f for f in FICHIERS if f["categorie"] == "temoin Windows sain"]
lignes = []

empreintes = {f["empreinte"] for f in modeles}
if len(empreintes) == 1:
    m = modeles[0]
    lignes.append("1. Les %d modeles compiles donnent au CNN exactement la meme image : "
                  "il ne peut pas les distinguer." % len(modeles))
    if temoin and temoin["empreinte"] == m["empreinte"]:
        lignes.append("2. Le programme vide compile avec PyInstaller donne AUSSI cette image. "
                      "Le CNN ne voit donc pas notre code, mais le lanceur de PyInstaller, "
                      "identique dans tout programme compile avec cet outil.")
    lignes.append("3. La fenetre analysee represente %.2f a %.2f %% de nos fichiers ; "
                  "notre code (quelques Ko compresses) se trouve bien plus loin dans le "
                  "fichier, dans l'archive ajoutee par PyInstaller."
                  % (min(f["part_vue_pct"] for f in modeles),
                     max(f["part_vue_pct"] for f in modeles)))
    lignes.append("4. Reponse du CNN pour ce lanceur : %s, confiance %.3f -> %s."
                  % (m["famille"], m["confiance"], m["verdict"]))
else:
    lignes.append("Les modeles compiles produisent %d images differentes :" % len(empreintes))
    for f in modeles:
        lignes.append("   %s : %s, confiance %.3f -> %s"
                      % (f["nom"], f["famille"], f["confiance"], f["verdict"]))

for f in sains:
    lignes.append("Temoin sain %s : %s, confiance %.3f -> %s."
                  % (f["nom"], f["famille"], f["confiance"], f["verdict"]))

lignes.append("")
lignes.append("Portee du test : ce CNN range un binaire dans l'une des 9 familles qu'il "
              "connait ; il n'a jamais vu de logiciel sain. Une confiance elevee signifie "
              "'ressemble a la famille X', pas 'est un virus'. Pour detecter un malware, "
              "il faudrait l'entrainer avec une dixieme classe de logiciels sains.")

CONCLUSION = "\n".join(lignes)
print(CONCLUSION)

In [ ]:
from google.colab import files
import shutil

TABLE.to_csv("/content/sorties/predictions.csv", index=False)
with open("/content/sorties/conclusion.txt", "w", encoding="utf-8") as f:
    f.write(CONCLUSION + "\n")
shutil.make_archive("/content/resultats_cnn_nos_modeles", "zip", "/content/sorties")
files.download("/content/resultats_cnn_nos_modeles.zip")